# 07 Occupancy Mapping From Synthetic Scans

This notebook extends the map-based visibility idea one step further:
instead of assuming the occupancy map is already known, we reconstruct an
occupancy grid from synthetic ray observations and then derive a
reliability field from the reconstructed map.

Demonstration question:
- If the camera only provides synthetic range-like rays, can we recover a map that is useful for visibility-aware planning?
- How different are **map reconstruction quality** and **planner-behavior quality**?
- When is a map-derived reliability field a useful comparison track, and when is it the wrong abstraction?

This is still a notebook-first methods study. It does **not** change the
runtime stack in `src/`. It also remains a comparison track rather than
the primary thesis mechanism. The primary thesis path is still the direct
state-dependent observation model from notebook `04`.


## Method, Formulas, Tradeoffs, And Why This Notebook Exists

Occupancy grids represent each cell `m_i` by an occupancy probability:

```math
p(m_i = 1) \in [0, 1].
```

A standard way to update this from range-style observations is the
**log-odds** form:

```math
L_t(m_i) = L_{t-1}(m_i) + \Delta L_t(m_i),
\qquad
p(m_i = 1) = \sigma(L_t(m_i)).
```

In this notebook the inverse sensor model is deliberately simple:
- cells traversed by a ray before the first hit receive a free-space increment `l_free`,
- the first hit cell receives an occupied-space increment `l_occ`,
- cells never touched by rays stay close to the prior `p = 0.5`.

Once a map is available, visibility is derived with the same soft
ray-based idea used in notebook `05`:

```math
q_{\mathrm{map}}(x) = \exp\left(-	au \; \overline{\mathrm{occ}}_{\mathrm{ray}}(x)
ight),
```

where `overline{occ}_ray(x)` is the mean occupancy sampled along the
camera-to-state ray. The planner then uses exactly the same mechanism as
notebook `04`:

```math
R_{\mathrm{eff}}(x) = q(x) R_{\mathrm{good}} + (1 - q(x)) R_{\mathrm{bad}}.
```

Why this notebook matters:
- it separates **map uncertainty** from the earlier direct learned-`q(x)` route,
- it shows a structure-aware reference model derived from environment geometry,
- it exposes a useful thesis distinction: a map can be mediocre in full-grid RMSE while still being good enough to recover planner-relevant visibility structure.

Advantages of occupancy maps:
- physically interpretable and easy to visualize,
- compatible with ray-based line-of-sight reasoning,
- a natural bridge to classical robotics literature.

Disadvantages:
- depends on the inverse sensor model and scan coverage,
- single-viewpoint mapping leaves large parts of the world weakly observed,
- occupancy alone cannot represent all appearance-dependent failures such as blur, boundary effects, or weak markers.

Alternatives and comparisons:
- hard line-of-sight on a known map: simpler, but brittle,
- GP occupancy maps: smoother and uncertainty-aware, but heavier,
- direct learned `q(x)`: bypasses explicit mapping and may fit the thesis question more directly,
- TSDF / ESDF style geometry models: useful for navigation, but less natural for detection reliability.


In [ ]:
from pathlib import Path
import sys
import math

repo_root = Path.cwd()
if not (repo_root / "scripts").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.state_dependent_observation_helpers import (
    OccupancyGrid2D,
    covariance_logdet_series,
    covariance_trace_series,
    evaluate_visibility_on_grid,
    grid_states,
    make_action_library,
    make_default_camera,
    make_occupancy_grid,
    make_observation_fn,
    observation_covariance,
    process_covariance_from_rho,
    raycast_visibility_q,
    score_action_library,
    simulate_receding_horizon,
)

rng = np.random.default_rng(9)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.22,
})


## Scenario And Modeling Assumptions

The setup mirrors the earlier notebooks:
- a fixed external camera looks into the workspace,
- the robot state is still planar: `x = [X, Y, theta]`,
- observations are still image-plane measurements through the notebook camera model,
- planning still uses ET-style nonlinear belief propagation through the helper layer.

The new ingredient is only the map reconstruction process.

Deliberate simplifications in this notebook:
- the scans are synthetic and noise-free in geometry,
- the inverse sensor model is hand-set rather than calibrated,
- there is only one camera/viewpoint,
- we reconstruct occupancy from random rays rather than running a full SLAM loop.

These choices are intentional. The goal is not to solve mapping fully, but
to isolate whether a learned map can supply a useful visibility field for
the planner.


In [ ]:
camera = make_default_camera()
camera_xy = np.asarray(camera.cam_pos[:2], dtype=float)

true_grid = make_occupancy_grid(
    xmin=-4.5,
    xmax=2.5,
    ymin=-4.8,
    ymax=0.8,
    resolution=0.1,
    rectangles=[
        (-0.75, 0.75, -2.60, -1.20),
        (-2.4, -1.8, -4.2, -3.3),
    ],
    circles=[(0.0, -3.7, 0.35)],
    border_occupancy=True,
)

goal_state = np.array([0.8, -1.5, 0.0], dtype=float)
representative_state = np.array([-0.8, -2.0, 0.0], dtype=float)

print("Camera xy:", camera_xy)
print("Map extent:", true_grid.extent)
print("True occupancy fraction:", float(true_grid.occupancy.mean()))


In [ ]:
def sample_occ(grid: OccupancyGrid2D, x: float, y: float) -> float:
    xs = grid.xs
    ys = grid.ys
    occ = grid.occupancy
    if x <= xs[0] or x >= xs[-1] or y <= ys[0] or y >= ys[-1]:
        return 1.0

    ix = int(np.searchsorted(xs, x, side="right") - 1)
    iy = int(np.searchsorted(ys, y, side="right") - 1)
    ix = max(0, min(ix, xs.shape[0] - 2))
    iy = max(0, min(iy, ys.shape[0] - 2))

    x0, x1 = xs[ix], xs[ix + 1]
    y0, y1 = ys[iy], ys[iy + 1]
    tx = 0.0 if x1 == x0 else (x - x0) / (x1 - x0)
    ty = 0.0 if y1 == y0 else (y - y0) / (y1 - y0)

    z00 = occ[iy, ix]
    z10 = occ[iy, ix + 1]
    z01 = occ[iy + 1, ix]
    z11 = occ[iy + 1, ix + 1]
    return float(
        (1.0 - ty) * ((1.0 - tx) * z00 + tx * z10)
        + ty * ((1.0 - tx) * z01 + tx * z11)
    )


def ray_samples(start, end, n=110):
    ts = np.linspace(0.0, 1.0, int(max(n, 2)))
    return np.outer(1.0 - ts, start) + np.outer(ts, end)


def first_hit_point(grid, start, end, threshold=0.55, n=110):
    points = ray_samples(start, end, n=n)
    occ_values = np.asarray([sample_occ(grid, p[0], p[1]) for p in points], dtype=float)
    hit_idx = np.flatnonzero(occ_values >= float(threshold))
    if hit_idx.size == 0:
        return None, points
    idx = int(hit_idx[0])
    return points[idx].copy(), points[: idx + 1].copy()


def world_to_cell(grid, x, y):
    ix = int(np.argmin(np.abs(grid.xs - x)))
    iy = int(np.argmin(np.abs(grid.ys - y)))
    return iy, ix


def update_log_odds(
    log_odds,
    visit_counts,
    grid,
    ray_points,
    hit_point,
    *,
    l_free=math.log(0.35 / 0.65),
    l_occ=math.log(0.80 / 0.20),
    clip=5.0,
):
    # All cells traversed before the first hit are treated as free.
    free_points = ray_points[:-1] if hit_point is not None else ray_points
    for point in free_points:
        iy, ix = world_to_cell(grid, point[0], point[1])
        log_odds[iy, ix] = np.clip(log_odds[iy, ix] + l_free, -clip, clip)
        visit_counts[iy, ix] += 1

    # The first hit cell is treated as occupied.
    if hit_point is not None:
        iy, ix = world_to_cell(grid, hit_point[0], hit_point[1])
        log_odds[iy, ix] = np.clip(log_odds[iy, ix] + l_occ, -clip, clip)
        visit_counts[iy, ix] += 1


def occ_from_log_odds(log_odds):
    return 1.0 / (1.0 + np.exp(-log_odds))


def copy_grid_with_occupancy(grid, occupancy):
    return OccupancyGrid2D(
        xs=grid.xs.copy(),
        ys=grid.ys.copy(),
        occupancy=np.asarray(occupancy, dtype=float).copy(),
        resolution=float(grid.resolution),
    )


def reconstruct_map_from_synthetic_scans(
    true_grid,
    camera_xy,
    rng,
    *,
    n_rays=260,
    n_samples=110,
    checkpoints=(25, 75, 150, 260),
    n_example_rays=28,
):
    log_odds = np.zeros_like(true_grid.occupancy, dtype=float)
    visit_counts = np.zeros_like(true_grid.occupancy, dtype=float)
    checkpoint_data = {}
    example_rays = []

    for step in range(1, int(n_rays) + 1):
        end = np.array([
            rng.uniform(true_grid.xs.min() + 0.1, true_grid.xs.max() - 0.1),
            rng.uniform(true_grid.ys.min() + 0.1, true_grid.ys.max() - 0.1),
        ], dtype=float)

        hit_point, traversed_points = first_hit_point(
            true_grid,
            camera_xy,
            end,
            threshold=0.55,
            n=n_samples,
        )
        update_log_odds(log_odds, visit_counts, true_grid, traversed_points, hit_point)

        if step <= int(n_example_rays):
            example_rays.append({
                "end": end.copy(),
                "hit_point": None if hit_point is None else hit_point.copy(),
                "ray_points": traversed_points.copy(),
            })

        if step in set(checkpoints):
            checkpoint_data[step] = {
                "occupancy": occ_from_log_odds(log_odds).copy(),
                "visit_counts": visit_counts.copy(),
            }

    return {
        "estimated_grid": copy_grid_with_occupancy(true_grid, occ_from_log_odds(log_odds)),
        "visit_counts": visit_counts,
        "checkpoints": checkpoint_data,
        "example_rays": example_rays,
    }


In [ ]:
checkpoint_steps = (25, 75, 150, 260)
mapping_run = reconstruct_map_from_synthetic_scans(
    true_grid,
    camera_xy,
    rng,
    n_rays=checkpoint_steps[-1],
    n_samples=110,
    checkpoints=checkpoint_steps,
    n_example_rays=28,
)

estimated_grid = mapping_run["estimated_grid"]
visit_counts = mapping_run["visit_counts"]
observed_mask = visit_counts > 0
example_rays = mapping_run["example_rays"]

print("Observed cell fraction:", float(observed_mask.mean()))
print("Final occupancy mean:", float(estimated_grid.occupancy.mean()))


In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 6.5), constrained_layout=True)
ax.imshow(true_grid.occupancy, origin="lower", extent=true_grid.extent, cmap="Greys")
ax.scatter(camera_xy[0], camera_xy[1], marker="^", s=120, color="tab:blue", label="camera")
ax.scatter(goal_state[0], goal_state[1], marker="*", s=180, color="tab:red", label="goal")

for idx, record in enumerate(example_rays):
    points = record["ray_points"]
    color = "tab:red" if record["hit_point"] is not None else "tab:green"
    alpha = 0.32 if idx < 10 else 0.16
    ax.plot(points[:, 0], points[:, 1], color=color, linewidth=1.5, alpha=alpha)
    if record["hit_point"] is not None:
        ax.scatter(record["hit_point"][0], record["hit_point"][1], color=color, s=20, alpha=0.55)

ax.set_title("True occupancy map with example synthetic rays")
ax.set_xlabel("world x [m]")
ax.set_ylabel("world y [m]")
ax.legend(loc="upper right")
plt.show()


## Reconstruction Snapshots

The next figure shows how the occupancy estimate evolves after increasing
numbers of rays. A useful detail to watch is that many cells remain near
`p = 0.5` for a long time. That is not necessarily a bug. With one fixed
camera, many cells are simply weakly observed or never traversed by any
informative ray.

This is why the notebook tracks two map metrics:
- **full-grid RMSE**: penalizes every weakly observed cell,
- **observed-region RMSE**: only evaluates cells that were actually touched by rays.

The latter often drops much faster, and it is usually the better predictor
of whether map-derived visibility will be planner-useful.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.5, 9.5), constrained_layout=True)

for ax, step in zip(axes.ravel(), checkpoint_steps):
    checkpoint = mapping_run["checkpoints"][step]
    occupancy = checkpoint["occupancy"]
    coverage = float((checkpoint["visit_counts"] > 0).mean())
    ax.imshow(occupancy, origin="lower", extent=true_grid.extent, vmin=0.0, vmax=1.0, cmap="viridis")
    ax.scatter(camera_xy[0], camera_xy[1], marker="^", s=90, color="white", edgecolor="black")
    ax.set_title(f"{step} rays | observed fraction={coverage:.2f}")
    ax.set_xlabel("world x [m]")
    ax.set_ylabel("world y [m]")

fig.suptitle("Occupancy reconstruction checkpoints", fontsize=14)
plt.show()


In [ ]:
xs_eval, ys_eval, states_eval = grid_states(
    xmin=true_grid.xs.min(),
    xmax=true_grid.xs.max(),
    ymin=true_grid.ys.min(),
    ymax=true_grid.ys.max(),
    nx=58,
    ny=46,
    theta=0.0,
)

q_true_fn = lambda state: raycast_visibility_q(state, true_grid, camera_xy, tau=8.0, n_samples=80)

rows = []
for step in checkpoint_steps:
    checkpoint = mapping_run["checkpoints"][step]
    occupancy = checkpoint["occupancy"]
    est_grid = copy_grid_with_occupancy(true_grid, occupancy)
    q_est_fn = lambda state, grid=est_grid: raycast_visibility_q(state, grid, camera_xy, tau=8.0, n_samples=80)

    q_true_map = evaluate_visibility_on_grid(states_eval, q_true_fn)
    q_est_map = evaluate_visibility_on_grid(states_eval, q_est_fn)
    observed = checkpoint["visit_counts"] > 0

    rows.append({
        "rays": step,
        "observed_fraction": float(observed.mean()),
        "occupancy_rmse_full": float(np.sqrt(np.mean((occupancy - true_grid.occupancy) ** 2))),
        "occupancy_rmse_observed": float(np.sqrt(np.mean((occupancy[observed] - true_grid.occupancy[observed]) ** 2))),
        "visibility_rmse": float(np.sqrt(np.mean((q_est_map - q_true_map) ** 2))),
        "visibility_corr": float(np.corrcoef(q_est_map.ravel(), q_true_map.ravel())[0, 1]),
    })

metrics_df = pd.DataFrame(rows)
print("Checkpoint metrics:")
metrics_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)

axes[0].plot(metrics_df["rays"], metrics_df["observed_fraction"], marker="o", linewidth=2)
axes[0].set_title("Observed map coverage")
axes[0].set_xlabel("number of rays")
axes[0].set_ylabel("fraction of cells touched")

axes[1].plot(metrics_df["rays"], metrics_df["occupancy_rmse_full"], marker="o", linewidth=2, label="full grid")
axes[1].plot(metrics_df["rays"], metrics_df["occupancy_rmse_observed"], marker="s", linewidth=2, label="observed cells")
axes[1].set_title("Occupancy RMSE")
axes[1].set_xlabel("number of rays")
axes[1].set_ylabel("RMSE")
axes[1].legend()

axes[2].plot(metrics_df["rays"], metrics_df["visibility_rmse"], marker="o", linewidth=2, label="visibility RMSE")
axes[2].plot(metrics_df["rays"], 1.0 - metrics_df["visibility_corr"], marker="s", linewidth=2, label="1 - correlation")
axes[2].set_title("Visibility error from estimated map")
axes[2].set_xlabel("number of rays")
axes[2].set_ylabel("error")
axes[2].legend()

plt.show()


## From Estimated Occupancy To A Reliability Field

The final reconstructed map is now turned into a planner-facing
reliability field `q_map(x)`.

This is a useful comparison because it reuses the same planner interface as
the earlier notebooks:
- notebook `04`: `q(x)` came from a hand-designed oracle field,
- notebook `06`: `q(x)` came from direct supervised learning,
- here: `q(x)` comes from a reconstructed occupancy map and ray-based visibility.

The key methodological point is that the final planner only sees `q(x)` and
`R_eff(x)`. It does not know whether `q(x)` came from an oracle field, a
learned classifier, or a map-derived visibility model.


In [ ]:
q_est_fn = lambda state: raycast_visibility_q(state, estimated_grid, camera_xy, tau=8.0, n_samples=80)
q_true_map = evaluate_visibility_on_grid(states_eval, q_true_fn)
q_est_map = evaluate_visibility_on_grid(states_eval, q_est_fn)
q_abs_error = np.abs(q_est_map - q_true_map)

x_slice = np.linspace(true_grid.xs.min(), true_grid.xs.max(), 180)
slice_states = np.stack([x_slice, np.full_like(x_slice, -2.0), np.zeros_like(x_slice)], axis=-1)
q_slice_true = np.asarray([q_true_fn(state) for state in slice_states], dtype=float)
q_slice_est = np.asarray([q_est_fn(state) for state in slice_states], dtype=float)

fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)

axes[0, 0].imshow(q_true_map, origin="lower", extent=true_grid.extent, vmin=0.0, vmax=1.0, cmap="viridis")
axes[0, 0].set_title("True-map soft visibility")
axes[0, 0].scatter(camera_xy[0], camera_xy[1], marker="^", s=90, color="white", edgecolor="black")

axes[0, 1].imshow(q_est_map, origin="lower", extent=true_grid.extent, vmin=0.0, vmax=1.0, cmap="viridis")
axes[0, 1].set_title("Estimated-map soft visibility")
axes[0, 1].scatter(camera_xy[0], camera_xy[1], marker="^", s=90, color="white", edgecolor="black")

im = axes[1, 0].imshow(q_abs_error, origin="lower", extent=true_grid.extent, vmin=0.0, vmax=max(0.15, float(q_abs_error.max())), cmap="magma")
axes[1, 0].set_title("Absolute visibility error")
axes[1, 0].scatter(camera_xy[0], camera_xy[1], marker="^", s=90, color="white", edgecolor="black")
plt.colorbar(im, ax=axes[1, 0], shrink=0.85)

axes[1, 1].plot(x_slice, q_slice_true, linewidth=2, label="true map")
axes[1, 1].plot(x_slice, q_slice_est, linewidth=2, linestyle="--", label="estimated map")
axes[1, 1].axvline(representative_state[0], color="black", linestyle=":", linewidth=1)
axes[1, 1].set_title("Visibility slice at y = -2.0 m")
axes[1, 1].set_xlabel("world x [m]")
axes[1, 1].set_ylabel("q(x)")
axes[1, 1].legend()

for ax in axes.ravel()[:3]:
    ax.set_xlabel("world x [m]")
    ax.set_ylabel("world y [m]")

plt.show()


## Planner Comparison

The planner comparison below answers a narrower question than the mapping
figures:
- even if the map is not perfect everywhere,
- does the **map-derived** reliability field reproduce the same first
  action as the true-map visibility field on states that matter?

This is intentionally the same logic as notebook `06`, where model quality
was judged not only by prediction error but also by **planner-behavior
agreement**.

Three planner variants are compared:
- constant `R`: ignores state-dependent visibility,
- true-map `q(x)`: uses visibility from the hidden ground-truth map,
- estimated-map `q(x)`: uses visibility from the reconstructed map.


In [ ]:
g = make_observation_fn(camera, obs_mode="uv")
R_good = observation_covariance("uv", uv_std=2.5)
R_bad = observation_covariance("uv", uv_std=20.0)
dt = 0.2
Q = process_covariance_from_rho(dt=dt, rho_xy=1e-2)
cov0 = np.diag([0.18 ** 2, 0.18 ** 2, 0.10 ** 2])
actions = make_action_library(
    v_values=(0.0, 0.22, 0.38),
    w_values=(-0.8, -0.35, 0.0, 0.35, 0.8),
)

common_kwargs = dict(
    dt=dt,
    Q=Q,
    g=g,
    goal_obs=g(goal_state),
    goal_obs_cov=np.diag([70.0 ** 2, 70.0 ** 2]),
    R_good=R_good,
    R_bad=R_bad,
    horizon=6,
    risk_weight=1.0,
    ambiguity_weight=0.7,
    control_weight=0.10,
    approx="ET2",
    add_ambiguity=True,
)

benchmark_states = np.array([
    [-0.8, -2.0, 0.0],
    [-0.2, -2.0, 0.0],
    [0.5, -2.2, 0.0],
    [0.7, -3.0, 0.0],
    [-1.1, -3.5, 0.0],
    [0.2, -1.5, 0.0],
], dtype=float)

benchmark_rows = []
for state in benchmark_states:
    constant_best = score_action_library(state, cov0, actions, q_fn=None, **common_kwargs)[0]
    true_best = score_action_library(state, cov0, actions, q_fn=q_true_fn, **common_kwargs)[0]
    est_best = score_action_library(state, cov0, actions, q_fn=q_est_fn, **common_kwargs)[0]
    benchmark_rows.append({
        "x": state[0],
        "y": state[1],
        "q_true": float(q_true_fn(state)),
        "q_est": float(q_est_fn(state)),
        "constant_action": tuple(np.round(constant_best.control, 2)),
        "true_map_action": tuple(np.round(true_best.control, 2)),
        "estimated_map_action": tuple(np.round(est_best.control, 2)),
        "estimated_matches_true": tuple(np.round(est_best.control, 2)) == tuple(np.round(true_best.control, 2)),
        "constant_matches_true": tuple(np.round(constant_best.control, 2)) == tuple(np.round(true_best.control, 2)),
    })

benchmark_df = pd.DataFrame(benchmark_rows)
planner_agreement_est = float(benchmark_df["estimated_matches_true"].mean())
planner_agreement_const = float(benchmark_df["constant_matches_true"].mean())

print("Estimated-map planner agreement with true map:", planner_agreement_est)
print("Constant-R planner agreement with true map:", planner_agreement_const)
benchmark_df


In [ ]:
mean0 = representative_state.copy()

runs = {
    "constant R": simulate_receding_horizon(
        mean0,
        cov0,
        actions,
        q_fn=None,
        n_steps=10,
        **common_kwargs,
    ),
    "true-map q(x)": simulate_receding_horizon(
        mean0,
        cov0,
        actions,
        q_fn=q_true_fn,
        n_steps=10,
        **common_kwargs,
    ),
    "estimated-map q(x)": simulate_receding_horizon(
        mean0,
        cov0,
        actions,
        q_fn=q_est_fn,
        n_steps=10,
        **common_kwargs,
    ),
}

first_action_df = pd.DataFrame([
    {
        "model": label,
        "first_action": tuple(np.round(run["controls"][0], 2)),
        "mean_q": float(run["q_values"].mean()) if len(run["q_values"]) else 1.0,
        "min_q": float(run["q_values"].min()) if len(run["q_values"]) else 1.0,
        "final_trace": float(covariance_trace_series(run["covs"])[-1]),
        "final_logdet": float(covariance_logdet_series(run["covs"])[-1]),
    }
    for label, run in runs.items()
])

print("Representative rollout summary:")
first_action_df


In [ ]:
q_background = q_true_map

fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)

axes[0, 0].imshow(q_background, origin="lower", extent=true_grid.extent, vmin=0.0, vmax=1.0, cmap="viridis", alpha=0.86)
axes[0, 0].imshow(true_grid.occupancy, origin="lower", extent=true_grid.extent, cmap="Greys", alpha=0.20)
for label, run in runs.items():
    axes[0, 0].plot(run["means"][:, 0], run["means"][:, 1], linewidth=2, label=label)
axes[0, 0].scatter(camera_xy[0], camera_xy[1], marker="^", s=90, color="white", edgecolor="black", label="camera")
axes[0, 0].scatter(goal_state[0], goal_state[1], marker="*", s=180, color="tab:red", label="goal")
axes[0, 0].scatter(mean0[0], mean0[1], marker="o", s=70, color="white", edgecolor="black", label="start")
axes[0, 0].set_title("Representative trajectories on true-map visibility")
axes[0, 0].set_xlabel("world x [m]")
axes[0, 0].set_ylabel("world y [m]")
axes[0, 0].legend(loc="lower right")

for label, run in runs.items():
    axes[0, 1].plot(run["q_values"], marker="o", linewidth=2, label=label)
axes[0, 1].set_title("Visibility along the rollout")
axes[0, 1].set_xlabel("step")
axes[0, 1].set_ylabel("q(x)")
axes[0, 1].legend()

for label, run in runs.items():
    axes[1, 0].plot(run["ambiguity_terms"], marker="o", linewidth=2, label=label)
axes[1, 0].set_title("Ambiguity term over time")
axes[1, 0].set_xlabel("step")
axes[1, 0].set_ylabel("ambiguity")
axes[1, 0].legend()

for label, run in runs.items():
    axes[1, 1].plot(covariance_trace_series(run["covs"]), marker="o", linewidth=2, label=label)
axes[1, 1].set_title("Belief covariance trace")
axes[1, 1].set_xlabel("step")
axes[1, 1].set_ylabel("trace(P)")
axes[1, 1].legend()

plt.show()


## Takeaways, Limitations, Alternatives, Sources

Main takeaways this notebook is meant to support:
- a partially reconstructed occupancy map can still recover planner-relevant visibility structure,
- full-grid occupancy RMSE can look pessimistic because many cells remain weakly observed from a single camera,
- planner agreement with the true-map visibility field can be substantially better than raw global map metrics suggest,
- map-derived visibility is a useful **comparison track**, but it is still not the cleanest primary mechanism for the thesis question.

Why it is not the primary thesis path:
- the thesis question is about whether state-dependent observation quality changes EFE behavior,
- direct `q(x)` in notebook `04` answers that question with fewer moving parts,
- occupancy mapping introduces extra assumptions about environment structure and the inverse sensor model.

Important limitations:
- one fixed camera causes sparse and anisotropic coverage,
- the inverse sensor model is hand-tuned rather than learned,
- scans are synthetic and do not include real missed detections or calibration drift,
- occupancy alone cannot represent all detector failure modes.

Useful alternatives to mention in the thesis:
- direct learned `q(x)` from detection outcomes,
- GP occupancy maps or sparse GP occupancy for smoother map inference,
- known-map hard or soft visibility as an oracle/reference baseline,
- multi-view mapping if the sensor platform changes over time.

Sources and anchors:
- A. Elfes, *Occupancy Grids: A Stochastic Spatial Representation for Active Robot Perception*, 1989.
- S. Thrun, W. Burgard, D. Fox, *Probabilistic Robotics*, 2005.
- Local repo anchors: `docs/math_to_code_map.md`, `docs/investigative_study_plan.md`, and `scripts/05_map_based_visibility_reference.ipynb`.
- Notebook helper implementation used here: `scripts/state_dependent_observation_helpers.py`.
